# Data Preprocessing

The objective of this notebook is to prepare the dataset for machine learning by handling missing values, encoding categorical variables, and creating a clean feature set for model training.

The preprocessing steps are applied consistently to both the training and test datasets to ensure reliable model performance.

In [59]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

### Load data

In [60]:
train = pd.read_csv("../data/train.csv")
test =pd.read_csv("../data/test.csv")

### Saparate target

In [61]:
X = train.drop(columns=["health_condition"])
y = train["health_condition"]

X_test = test.copy()

### Identify features

In [62]:
num_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()

cat_cols = X.select_dtypes(include="object").columns.tolist()

print(num_cols)
print(cat_cols)

['id', 'sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


In [63]:
print(cat_cols)

for col in cat_cols:
    print(f"\n{col}")
    print(X[col].unique())

['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

diet_type
['veg' 'non-veg' 'balanced' nan]

stress_level
['high' 'low' nan 'medium']

sleep_quality
['average' 'poor' nan 'good']

physical_activity_level
['sedentary' 'moderate' 'active' nan]

smoking_alcohol
['yes' 'occasional' nan 'no']

gender
['female' 'other' 'male' nan]


### Missing Value Imputation

In [64]:
# Numerical
num_imputer = SimpleImputer(strategy="median")
X[num_cols] = num_imputer.fit_transform(X[num_cols])
X_test[num_cols] = num_imputer.fit_transform(X_test[num_cols])

In [65]:
# Categorical
cat_imputer = SimpleImputer(strategy="most_frequent")
X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
X_test[cat_cols] = cat_imputer.fit_transform(X_test[cat_cols])

### Check Missing

In [66]:
print(X.isnull().sum().sum())

print(X_test.isnull().sum().sum())

0
0


### Encode Categorical

In [67]:
# Ordinal features
ordinal_maps = {
    "stress_level": {
        "low": 0,
        "medium": 1,
        "high": 2
    },

    "sleep_quality": {
        "poor": 0,
        "average": 1,
        "good": 2
    },

    "physical_activity_level": {
        "sedentary": 0,
        "moderate": 1,
        "active": 2
    }
}

for col, mapping in ordinal_maps.items():
    X[col] = X[col].map(mapping)
    X_test[col] = X_test[col].map(mapping)

In [68]:
# Nominal features
nominal_cols = [
    "diet_type",
    "smoking_alcohol",
    "gender"
]

X = pd.get_dummies(
    X,
    columns=nominal_cols,
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    columns=nominal_cols,
    drop_first=True
)

In [69]:
X, X_test = X.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

### Encode Target

In [70]:
mapping = {
    "fit":0,
    "at-risk":1,
    "unhealthy":2
}
y = y.map(mapping)

### final check

In [71]:
print("Missing values (train):", X.isnull().sum().sum())
print("Missing values (test):", X_test.isnull().sum().sum())

print("Train shape:", X.shape)
print("Test shape:", X_test.shape)

X.head()

Missing values (train): 0
Missing values (test): 0
Train shape: (690088, 17)
Test shape: (295753, 17)


,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,stress_level,sleep_quality,physical_activity_level,diet_type_non-veg,diet_type_veg,smoking_alcohol_occasional,smoking_alcohol_yes,gender_male,gender_other
0,0.0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,2,1,0,False,True,False,True,False,False
1,1.0,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,0,1,1,True,False,False,True,False,True
2,2.0,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,2,0,2,False,True,False,True,True,False
3,3.0,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,2,1,2,False,True,True,False,False,False
4,4.0,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,1,1,0,False,True,False,True,True,False


### Save

In [73]:
processed_train = X.copy()
processed_train["health_condition"] = y

processed_test = X_test.copy()

processed_train.to_csv("../data/processed/processed_train.csv", index=False)
processed_test.to_csv("../data/processed/processed_test.csv", index=False)

## Summary

The preprocessing pipeline successfully prepared the dataset for machine learning by:

- imputing missing numerical values using the median,
- imputing missing categorical values using the most frequent category,
- encoding ordinal features while preserving their natural order,
- applying one-hot encoding to nominal features,
- encoding the target variable,
- ensuring that the training and testing datasets share the same feature space.

The processed datasets are now ready for model development and evaluation.